# 🔭 Notebook 1: The Three Pillars — Logs, Metrics, Traces

Imagine you run a website and a customer tweets *"the checkout page is broken!"*. You jump to your terminal — what do you look at?

Real production systems use three complementary tools. Each is bad at the others' job. Together they form **observability**.

| Pillar | Question it answers | Best for |
|---|---|---|
| 🪵 **Logs** | *What happened, in detail, in chronological order?* | Debugging a single request |
| 📊 **Metrics** | *How is the system doing right now, in aggregate?* | Dashboards & alerts |
| 🧵 **Traces** | *Where did this one request spend its time across services?* | Finding latency bottlenecks |

> **Monitoring vs Observability** — *monitoring* tells you **whether** the system is healthy (a known-bad happens, an alert fires). *Observability* lets you ask **why** without redeploying — even for failures you didn't predict.

## Learning objectives
- Generate logs, metrics, and traces with nothing but stdlib + dicts.
- Tell **bad logs** from **good (structured) logs**.
- See why a single shared `request_id` (a.k.a. `trace_id`) ties all three pillars together.

## 🛠️ Setup

```bash
cd 01-foundations/observability
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). Reload the window if it doesn't appear (`Cmd+Shift+P` → **Reload Window**).

There are no external services — everything runs in-process.

## 🪵 Logs — the diary

A log line is timestamped text describing one event. Logs are great for *"what exactly happened to **this** request?"* — bad for *"how is the system doing across millions of requests?"* because reading them all is slow.

### ❌ Bad logs — unstructured prose

This is what you get if you sprinkle `print(...)` everywhere. It's readable for one person staring at a terminal, but **impossible to search, filter, or aggregate** at scale.

In [ ]:
# ❌ Bad: free-form text. Try grepping for "request 7c2a9e" — you'll be sad.
import time

print(f"[{time.strftime('%H:%M:%S')}] Got a request for /checkout from a user")
print(f"[{time.strftime('%H:%M:%S')}] Looked up the order in the DB, took a while")
print(f"[{time.strftime('%H:%M:%S')}] Payment failed because the card was declined I think")
print(f"[{time.strftime('%H:%M:%S')}] Returned 402 to the user")

### ✅ Good logs — structured (JSON) with a shared `request_id`

Every line is a JSON object with the same fields. A log shipper (Loki, Elasticsearch, CloudWatch...) can index it. You can filter by `req_id`, by `level`, or aggregate `ms` per `event`.

The single most important field is **`req_id`** — a unique ID per request that you also put in your traces and your error messages. That's how you jump from "one slow trace" to "the exact log lines for it".

In [ ]:
import time, json, uuid

def log(level, event, **fields):
    record = {
        "ts": time.strftime("%H:%M:%S"),
        "level": level,
        "event": event,
        **fields,
    }
    print(json.dumps(record))

# This req_id will travel through every log line, every metric label,
# and every trace span — that's how the three pillars correlate.
req_id = uuid.uuid4().hex[:8]

log("INFO",  "request.start", req_id=req_id, path="/checkout", user="alice")
log("INFO",  "db.query",      req_id=req_id, table="orders", ms=12)
log("ERROR", "payment.failed", req_id=req_id, reason="card_declined", provider="stripe")
log("INFO",  "request.end",   req_id=req_id, status=402, ms=87)

**Why structured?** Try answering *"how many `payment.failed` events with reason=`card_declined` happened in the last hour?"* with the bad logs. Now try with the good ones — it's a one-line query.

## 📊 Metrics — the dashboard

A metric is a single **number** sampled over time. There are three classic kinds:

- **Counter** — only goes up (e.g. `http_requests_total`). Reset on process restart.
- **Gauge** — goes up and down (e.g. `memory_in_use_bytes`, `queue_depth`).
- **Histogram** — buckets of observations, used for things like latency.

Metrics are tiny, cheap, and fast to query → perfect for **alerting** and **dashboards**. They cannot tell you *which* request was slow — only that *5%* of them were.

In [ ]:
import random

class Metrics:
    def __init__(self):
        self.counters = {}
        self.gauges = {}
        self.histograms = {}

    def inc(self, name, by=1):
        self.counters[name] = self.counters.get(name, 0) + by

    def gauge(self, name, value):
        self.gauges[name] = value

    def observe(self, name, value):
        self.histograms.setdefault(name, []).append(value)

m = Metrics()
random.seed(0)
for _ in range(1000):
    latency = random.gauss(50, 10)
    m.observe("http_latency_ms", latency)
    m.inc("http_requests_total")
    if latency > 70:
        m.inc("http_requests_slow_total")

m.gauge("memory_in_use_mb", 412)

vals = sorted(m.histograms["http_latency_ms"])
p50, p95, p99 = vals[500], vals[950], vals[990]
print(f"requests: {m.counters['http_requests_total']}")
print(f"slow:     {m.counters['http_requests_slow_total']}")
print(f"p50={p50:.1f} ms  p95={p95:.1f} ms  p99={p99:.1f} ms")
print(f"memory:   {m.gauges['memory_in_use_mb']} MB")

assert p50 <= p95 <= p99
assert m.counters["http_requests_total"] == 1000

### Why percentiles, not averages?

The **average** latency hides outliers. Real teams alert on **p95** or **p99**, not on
`avg`. Let's not take that on faith — here is the classic 99-happy-users-and-one-victim
distribution, measured.

In [ ]:
import statistics

# 1000 requests: 98% are snappy, 2% hit a 5-second timeout.
latencies = [50.0] * 980 + [5000.0] * 20

s = sorted(latencies)
avg = statistics.mean(s)
p50 = s[499]          # nearest-rank: 50% of 1000
p99 = s[989]          # nearest-rank: 99% of 1000

print(f"avg = {avg:7.1f} ms   <- 'the service is a bit slow, but fine'")
print(f"p50 = {p50:7.1f} ms   <- what a typical user sees")
print(f"p99 = {p99:7.1f} ms   <- what your angriest 1% see")

# The average sits between the two groups and describes NEITHER of them: it is
# 3x the median and 34x smaller than the tail. That is why it makes a bad alert.
assert avg > 2 * p50
assert p99 > 20 * avg
print(f"\nNobody experienced {avg:.0f} ms. The average is an artefact of mixing "
      "two populations.")

## 🧵 Traces — the breadcrumb trail

A **trace** follows one request as it bounces between services. Each step is a **span** with: a name, a start time, a duration, and a parent span. Together spans form a tree.

Traces are how you answer *"where is the time going?"* in a microservices architecture.

In [ ]:
class Span:
    def __init__(self, name, parent=None, trace_id=None):
        self.name = name
        self.parent = parent
        self.trace_id = trace_id or (parent.trace_id if parent else uuid.uuid4().hex[:8])
        self.start = time.perf_counter()
        self.duration_ms = 0.0
        self.children = []
        if parent:
            parent.children.append(self)

    def __enter__(self): return self
    def __exit__(self, *a):
        self.duration_ms = (time.perf_counter() - self.start) * 1000

    def show(self, indent=0):
        print(f"{' ' * indent}└─ {self.name:<22} {self.duration_ms:6.1f} ms  (trace={self.trace_id})")
        for c in self.children:
            c.show(indent + 4)

with Span("GET /checkout") as root:
    time.sleep(0.005)
    with Span("auth.verify_jwt", parent=root):
        time.sleep(0.002)
    with Span("orders.create", parent=root) as orders:
        with Span("db.insert", parent=orders):
            time.sleep(0.020)
        with Span("payments.charge", parent=orders):
            time.sleep(0.080)         # 👀 the slow one!
    with Span("send_email", parent=root):
        time.sleep(0.003)

root.show()

Reading the tree top-down, you can immediately see **`payments.charge` is dominating the request time** (~80 ms of ~110 ms). That's where to focus optimization. Without traces you'd just know "checkout is slow" — not *which step*.

## 🔗 The correlation trick — one ID to rule them all

The most underrated practice in observability:

> **Use the same ID as the `req_id` in your logs, the `trace_id` in your spans, and a label/header in any downstream call.**

That way, when an alert fires (metrics → "p99 latency spiked"), you can:
1. Pick a slow trace from your tracing UI.
2. Copy its `trace_id`.
3. Paste it into your log search → see the exact log lines for that one request.

Three tools, one investigation.

In [ ]:
# A tiny demo: one request, all three pillars sharing a trace_id.
trace_id = uuid.uuid4().hex[:8]

m = Metrics()
m.inc("http_requests_total")

with Span("GET /checkout", trace_id=trace_id) as root:
    log("INFO", "request.start", req_id=trace_id, path="/checkout")
    time.sleep(0.01)
    with Span("db.query", parent=root):
        time.sleep(0.015)
        log("INFO", "db.query", req_id=trace_id, table="orders", ms=15)
    log("INFO", "request.end", req_id=trace_id, status=200)

m.observe("http_latency_ms", root.duration_ms)
print()
root.show()
print(f"\nmetric http_requests_total = {m.counters['http_requests_total']}")
print(f"metric http_latency_ms (last) = {root.duration_ms:.1f} ms")

# The whole point of the exercise: the id in the span tree and the id in every log
# line are the SAME string, so a log search on it returns exactly this request.
assert root.trace_id == trace_id
assert all(c.trace_id == trace_id for c in root.children)

## 🎚️ One more axis: sampling

Metrics are cheap because they are already aggregated — a counter costs the same
whether it counted 10 requests or 10 million. **Traces are not.** A full trace is
kilobytes of spans per request, so nobody stores them all. You *sample*, and the
sampling strategy decides which failures you can still investigate:

| Strategy | How it works | Catches the rare error? | Cost |
|---|---|---|---|
| **Head sampling** | decide at the first span, e.g. "keep 1%" | ❌ only 1% of the time | cheap, decided up front |
| **Tail sampling** | buffer the whole trace, keep it if it was slow or errored | ✅ by construction | needs a buffer + a collector |

Head sampling is the default because it is trivial to implement. It is also the
reason on-call engineers so often find that the one trace they needed was thrown
away. Let's measure that.

In [ ]:
import random

random.seed(7)
N = 100_000
ERROR_RATE = 0.001          # 1 request in 1000 fails
HEAD_SAMPLE_RATE = 0.01     # keep 1% of traces

requests = [{"error": random.random() < ERROR_RATE} for _ in range(N)]
total_errors = sum(r["error"] for r in requests)

# Head sampling: the keep/drop coin is flipped before we know how it went.
head_kept = [r for r in requests if random.random() < HEAD_SAMPLE_RATE]
head_errors = sum(r["error"] for r in head_kept)

# Tail sampling: keep 1% of the boring ones, but keep every failure.
tail_kept = [r for r in requests if r["error"] or random.random() < HEAD_SAMPLE_RATE]
tail_errors = sum(r["error"] for r in tail_kept)

print(f"{total_errors} failing requests out of {N:,}\n")
print(f"head sampling: stored {len(head_kept):>6,} traces, "
      f"kept {head_errors:>3}/{total_errors} of the errors")
print(f"tail sampling: stored {len(tail_kept):>6,} traces, "
      f"kept {tail_errors:>3}/{total_errors} of the errors")

# Tail sampling keeps every error for almost exactly the same storage bill.
assert tail_errors == total_errors
assert head_errors < 0.2 * total_errors
assert len(tail_kept) < 1.2 * len(head_kept)
print(f"\nSame storage (+{len(tail_kept)/len(head_kept) - 1:.0%}), "
      f"{total_errors - head_errors} more debuggable failures.")

## 🤔 When to use which

| Question | Best tool |
|---|---|
| Is the system healthy right now? Are we in SLO? | **Metrics** |
| Why was *this specific request* slow / wrong? | **Traces** → then **Logs** |
| What was the exact error / stack trace for request X? | **Logs** |

In the next notebook we turn metrics into a contract: the **SLI / SLO / SLA** triangle and the **error budget** that comes out of it.